In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import sys

In [2]:
# 参考リンク : https://dev.syosetu.com/man/api/ 

# APIのエンドポイント
NAROU_API_URL = "https://api.syosetu.com/novelapi/api/"
# 1リクエストあたりの待機時間（秒）
REQUEST_INTERVAL = 1

In [3]:
def get_narou_data(start_index=1, limit=500, order="hyoka"):
    """
    なろう小説APIから指定された条件で小説データを取得します。
    """
    params = {
        "out": "json",
        "lim": limit,
        "st": start_index,
        "order": order,
        "of": "n-t-s-w-k-gl-l-nt-e-a-f-gf-ga"
    }

    try:
        response = requests.get(NAROU_API_URL, params=params)
        response.raise_for_status()
        data = response.json()
        
        if isinstance(data, list) and len(data) > 1 and isinstance(data[1], dict):
            return data[1:]
        else:
            print(f"警告: 小説データが見つかりませんでした。API応答: {response.text[:200]}...")
            return []
            
    except requests.exceptions.RequestException as e:
        print(f"APIリクエスト中にエラーが発生しました: {e}")
    except ValueError as e:
        print(f"API応答のJSON解析に失敗しました: {e}")
        
    return None

In [ ]:
def create_corpus(order_list, num_per_order=2000, output_filename="narou_corpus.csv"):
    """
    複数の並び順で小説データを収集し、重複を削除して保存します。
    """
    combined_novel_data = pd.DataFrame()
    limit = 500
    
    print(f"複数の並び順でデータ収集を開始します。各並び順で最大{num_per_order}件取得します。")

    for order in order_list:
        all_novel_data = []
        max_requests = (num_per_order // limit) + (1 if num_per_order % limit > 0 else 0)

        print(f"\n--- {order}順でのデータ収集 ({num_per_order}件) ---")
        for i in range(max_requests):
            start = i * limit + 1
            if order == "hyoka" and start > 2000:
                print(f"警告: 評価順では2000件以降は取得できません。{start}件目以降の取得をスキップします。")
                break

            print(f"{i+1}回目のリクエスト (通算{start}件目～)...")
            novels = get_narou_data(start_index=start, limit=limit, order=order)

            if novels is None:
                print("APIリクエストに失敗したため、この並び順での処理を中断します。")
                break
            
            if not novels:
                print("これ以上、有効な小説データを取得できませんでした。")
                break

            all_novel_data.extend(novels)
            
            if len(all_novel_data) >= num_per_order:
                print(f"目標件数（{num_per_order}件）を達成しました。")
                break
            
            time.sleep(REQUEST_INTERVAL)

        if all_novel_data:
            df_temp = pd.DataFrame(all_novel_data)
            combined_novel_data = pd.concat([combined_novel_data, df_temp], ignore_index=True)
            print(f"{order}順で{len(df_temp)}件のデータを取得しました。")
            print(f"現在の合計件数（重複あり）: {len(combined_novel_data)}件")
            
    if combined_novel_data.empty:
        print("\n小説データを1件も取得できませんでした。")
        return

    # 重複の削除
    print(f"\n合計 {len(combined_novel_data)}件 のデータから重複を削除します...")
    df = combined_novel_data.drop_duplicates(subset=['ncode'], keep='first').copy()
    print(f"重複削除後、{len(df)}件のユニークな小説データになりました。")

    # データフレームの処理（リネーム、更新頻度計算など）
    df.rename(columns={
        'ncode': 'Nコード', 'title': 'タイトル', 'story': 'あらすじ',
        'writer': '作者', 'keyword': 'キーワード', 'general_lastup': '最終更新日',
        'general_firstup': '初回投稿日', 'general_all_no': '投稿話数',
        'length': '文字数', 'noveltype': '小説タイプ', 'end': '完結フラグ',
        'all_point': '総合評価ポイント', 'fav_novel_cnt': 'ブックマーク数'
    }, inplace=True)
    
    required_columns = ['最終更新日', '初回投稿日', '投稿話数']
    for col in required_columns:
        if col not in df.columns:
            print(f"\nエラー: '{col}' のデータが取得できませんでした。")
            sys.exit()

    df['完結フラグ'] = pd.to_numeric(df['完結フラグ'], errors='coerce').fillna(0).astype(int)
    df['最終更新日_dt'] = pd.to_datetime(df['最終更新日'])
    df['初回投稿日_dt'] = pd.to_datetime(df['初回投稿日'])

    df['投稿期間_日数'] = (df['最終更新日_dt'] - df['初回投稿日_dt']).dt.days
    df['更新頻度_日数/話'] = df.apply(
        lambda row: row['投稿期間_日数'] / (row['投稿話数'] - 1) if row['投稿話数'] > 1 else (0 if row['投稿話数'] == 1 and row['投稿期間_日数'] == 0 else None),
        axis=1
    )

    one_year_ago = datetime.now() - timedelta(days=365)
    is_eternal_condition = (df['完結フラグ'] == 0) & (df['最終更新日_dt'] < one_year_ago)
    df['is_eternal'] = is_eternal_condition.astype(int)
    
    print(f"\nフィルタリング前のデータ件数: {len(df)}件")
    
    # 「連載中」かつ「1年以内に更新がある」作品を除外する条件
    condition_to_exclude = (df['完結フラグ'] == 0) & (df['is_eternal'] == 0)
    
    num_before_filter = len(df)
    df = df[~condition_to_exclude].copy() # 条件に合致しない行、つまり完結済みか長期未更新の作品のみを残す
    num_after_filter = len(df)
    
    print(f"フィルタリングにより、アクティブな連載作品 {num_before_filter - num_after_filter}件 を除外しました。")
    print(f"フィルタリング後のデータ件数: {num_after_filter}件")
    # ▲▲▲ 変更点：ここまで ▲▲▲

    df = df.drop(columns=['最終更新日_dt', '初回投稿日_dt', '投稿期間_日数'])

    df.to_csv("corpus/"+output_filename, index=False, encoding='utf-8-sig')
    print(f"\nデータ収集が完了し、'{output_filename}' に保存しました。")
    print("\n--- データの内訳 ---")
    print(df['is_eternal'].value_counts())
    print("--------------------")

In [5]:
NUM_OF_NOVELS_TO_FETCH = 2000
order_to_fetch = ["new","favnovelcnt","reviewcnt","hyoka","hyokaasc","dailypoint","weeklypoint","monthlypoint","quarterpoint","yearlypoint","impressioncnt","hyokacnt","hyokacntasc","weekly","lengthdesc","lengthasc","generalfirstup","ncodeasc","ncodedesc","old"]
create_corpus(order_to_fetch, num_per_order=NUM_OF_NOVELS_TO_FETCH)

複数の並び順でデータ収集を開始します。各並び順で最大2000件取得します。

--- new順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
目標件数（2000件）を達成しました。
new順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 2000件

--- favnovelcnt順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
目標件数（2000件）を達成しました。
favnovelcnt順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 4000件

--- reviewcnt順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
目標件数（2000件）を達成しました。
reviewcnt順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 6000件

--- hyoka順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
目標件数（2000件）を達成しました。
hyoka順で2000件のデータを取得しました。
現在の合計件数（重複あり）: 8000件

--- hyokaasc順でのデータ収集 (2000件) ---
1回目のリクエスト (通算1件目～)...
2回目のリクエスト (通算501件目～)...
3回目のリクエスト (通算1001件目～)...
4回目のリクエスト (通算1501件目～)...
目標件数（2000件）を達成しました。
hyokaasc順で2000件のデータを取得しま